#### 导入包

In [15]:
# 导入所有必要的库
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import requests
from PIL import Image
# import cv2
import os
import re
from sklearn.model_selection import train_test_split
# 用于文本处理
from emoji import demojize
import opencc
# 用于CLIP分词 (这里以transformers库中的中文CLIP为例)
from transformers import ChineseCLIPProcessor
# 用于图像变换
import torchvision.transforms as transforms
# 设置随机种子保证可复现
torch.manual_seed(42)
np.random.seed(42)

#### 初步分析数据

In [ ]:
# 加载数据
df_train = pd.read_csv('/kaggle/input/datasets/alettatta/frd-dataset/train_ready.csv')
print("\n前5行数据:")
display(df_train.head())
print(df_train['label'].value_counts())

In [ ]:
df_val = pd.read_csv('/kaggle/input/datasets/alettatta/frd-dataset/val_ready.csv')

In [ ]:
df_test = pd.read_csv("/kaggle/input/datasets/alettatta/frd-dataset/test_ready.csv")

In [ ]:
df_test.columns

### 图像清洗

In [ ]:
# 检查
import clip
print(clip.__file__)

In [ ]:
import os
import requests
import pandas as pd
from tqdm import tqdm
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- 关键修改：Kaggle 可写路径 ---
IMG_SAVE_DIR = "/kaggle/working/images"
os.makedirs(IMG_SAVE_DIR, exist_ok=True)

HEADERS_LIST = [
    {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"},
    {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"},
    {"User-Agent": "Mozilla/5.0", "Referer": "https://www.google.com/"}
]

def normalize_url(url: str) -> str:
    if pd.isna(url) or not isinstance(url, str):
        return None
    url = url.strip()
    if url.startswith("//"):
        return "https:" + url
    return url

def infer_ext_from_content_type(content_type: str) -> str:
    content_type = content_type.lower()
    if "jpeg" in content_type or "jpg" in content_type: return ".jpg"
    if "png" in content_type: return ".png"
    if "webp" in content_type: return ".webp"
    return ".jpg"

def download_image(url: str) -> str | None:
    url = normalize_url(url)
    if not url: return None

    # 使用 MD5 避免重复下载
    filename_base = hashlib.md5(url.encode("utf-8")).hexdigest()
    
    # 预检：如果任意格式的该文件已存在，直接返回（节省流量和时间）
    for ext in [".jpg", ".png", ".webp"]:
        potential_path = os.path.join(IMG_SAVE_DIR, filename_base + ext)
        if os.path.exists(potential_path):
            return potential_path

    for headers in HEADERS_LIST:
        try:
            r = requests.get(url, headers=headers, timeout=10, stream=True)
            if r.status_code == 200:
                content_type = r.headers.get("Content-Type", "")
                ext = infer_ext_from_content_type(content_type)
                save_path = os.path.join(IMG_SAVE_DIR, filename_base + ext)
                
                with open(save_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=8192):
                        f.write(chunk)
                return save_path
        except:
            continue
    return None

# --- 关键修改：多线程下载函数 ---
def parallel_download(df, url_col="image_url", max_workers=8):
    urls = df[url_col].tolist()
    results = [None] * len(urls)
    
    print(f"开始并行下载 {len(urls)} 张图片...")
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # 建立 索引->Future 的映射
        future_to_idx = {executor.submit(download_image, url): i for i, url in enumerate(urls)}
        
        for future in tqdm(as_completed(future_to_idx), total=len(urls), desc="Downloading"):
            idx = future_to_idx[future]
            try:
                results[idx] = future.result()
            except Exception:
                results[idx] = None
    return results

In [ ]:
dataframes = {"train": df_train, "val": df_val, "test": df_test}

for name, df in dataframes.items():
    print(f"\n正在处理 {name} 集...")
    # 使用多线程下载
    df["img_local_path"] = parallel_download(df)
    
    # 统计
    success_rate = df["img_local_path"].notna().mean()
    print(f"{name} 集下载成功率: {success_rate:.2%}")

# 最后检查一下路径
print(f"\n示例路径: {df_train['img_local_path'].iloc[0]}")

#### 多模态 Dataset

In [16]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor

# ====================================================
# 1. 数据清洗：移除下载失败的样本 (极其重要)
# ====================================================
print("原始数据量:", len(df_train), len(df_val), len(df_test))

原始数据量: 5453 1168 1169


In [17]:
# ====================================================
# 2. 加载 CLIP Processor (在线下载或从 Kaggle Dataset 加载)
# ====================================================
# 注意：确保 Kaggle 右侧 Settings 的 Internet 为 On
MODEL_NAME = "openai/clip-vit-base-patch32"

try:
    processor = AutoProcessor.from_pretrained(MODEL_NAME)
    print("CLIP Processor 加载成功")
except Exception as e:
    print(f"加载失败，请检查网络设置或路径: {e}")

image_transform = processor.image_processor

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


CLIP Processor 加载成功


In [13]:
# ====================================================
# 3. 定义图像加载辅助函数 (带容错)
# ====================================================
def load_and_transform_image(img_path, transform):
    try:
        image = Image.open(img_path).convert("RGB")
        # CLIP 预处理器返回的是字典，提取 ['pixel_values']
        pixel_values = transform(images=image, return_tensors="pt")["pixel_values"]
        return pixel_values.squeeze(0) # [3, 224, 224]
    except Exception as e:
        # 万一读取出错，返回全黑张量
        return torch.zeros((3, 224, 224))

# ====================================================
# 4. 定义 Multimodal Dataset
# ====================================================
class MultimodalReviewDataset(Dataset):
    def __init__(self, dataframe, processor, image_transform):
        self.df = dataframe.reset_index(drop=True)
        self.processor = processor
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # --- 文本处理 ---
        # CLIP 默认 max_length 为 77
        text_inputs = self.processor(
            text=str(row["text_clean"]),
            truncation=True,
            max_length=77,
            padding="max_length",
            return_tensors="pt"
        )
        
        # --- 图像处理 ---
        img_tensor = load_and_transform_image(
            row["img_local_path"], 
            self.image_transform
        )

        label = torch.tensor(row["label"], dtype=torch.long)

        return {
            "text": text_inputs["input_ids"].squeeze(0),
            "attention_mask": text_inputs["attention_mask"].squeeze(0),
            "image": img_tensor,
            "label": label
        }

In [ ]:
# ====================================================
# 5. 构建 Dataset 和 DataLoader
# ====================================================
train_dataset = MultimodalReviewDataset(df_train, processor, image_transform)
val_dataset   = MultimodalReviewDataset(df_val, processor, image_transform)
test_dataset  = MultimodalReviewDataset(df_test, processor, image_transform)

# 建议：如果类别极度不平衡，再考虑使用 weighted_sampler
# 这里先使用默认的 shuffle
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4, # Kaggle 建议设为 2 或 4
    pin_memory=True
)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print(f"DataLoader Ready! Train Batches: {len(train_loader)}")

In [ ]:
# ====================================================
# 6. 保存 DataFrame 进度 (存入 Kaggle 可写目录)
# ====================================================
SAVE_PATH = "/kaggle/working/tmp"
os.makedirs(SAVE_PATH, exist_ok=True)

df_train.to_pickle(f"{SAVE_PATH}/train_processed.pkl")
df_val.to_pickle(f"{SAVE_PATH}/val_processed.pkl")
df_test.to_pickle(f"{SAVE_PATH}/test_processed.pkl")

print(f"进度已保存至 {SAVE_PATH}")

#### 训练导入包

In [4]:
import os
import torch
from torch.utils.data import DataLoader, Dataset
from torch import nn, optim
from tqdm import tqdm
import pickle

In [5]:
import os
import sys

# 源文件路径
src_path = "/kaggle/input/models/alettatta/frd/pytorch/default/1/multimodal_model_new(final).py"
# 目标路径（存放在当前工作目录，起一个规范的名字）
dst_path = "/kaggle/working/multimodal_model.py"

# 创建软链接（如果不存在）
if not os.path.exists(dst_path):
    os.symlink(src_path, dst_path)

# 将当前目录加入路径
sys.path.append("/kaggle/working")

# 现在可以正常 import 了
from multimodal_model import ECommerceMultimodalModel

In [ ]:
import torch

# 1. 检查CUDA是否可用
is_available = torch.cuda.is_available()
print(f"CUDA is available: {is_available}")

if is_available:
    # 2. 获取GPU的数量
    gpu_count = torch.cuda.device_count()
    print(f"Number of GPUs available: {gpu_count}")

    # 3. 获取当前GPU的名称
    current_gpu_name = torch.cuda.get_device_name(0)
    print(f"Name of the current GPU: {current_gpu_name}")

    # 4. 获取当前GPU的设备ID
    current_device = torch.cuda.current_device()
    print(f"ID of the current GPU: {current_device}")
else:
    print("PyTorch was installed with GPU support, but no compatible NVIDIA GPU/driver was found.")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16
EPOCHS = 30

In [ ]:
# 加载clip
import clip
from PIL import Image

clip_model, preprocess = clip.load("ViT-B/32", device=device)

In [10]:
from transformers import BertTokenizer

# 自动从 Hugging Face 官网下载对应版本的配置和词表
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

class ReviewDataset(Dataset):
    def __init__(self, df, preprocess):
        self.df = df.reset_index(drop=True)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = row["text_clean"]

        # 2. 用分词器编码文本
        bert_enc = bert_tokenizer(
            text,
            max_length=77,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        bert_input_ids = bert_enc["input_ids"].squeeze(0)        # [77]
        bert_attention_mask = bert_enc["attention_mask"].squeeze(0)

        # CLIP 部分保持不变
        clip_input_ids = clip.tokenize([text], truncate=True).squeeze(0)

        image = Image.open(row["img_local_path"]).convert("RGB")
        image_tensor = self.preprocess(image)

        return {
            "bert_input_ids": bert_input_ids,
            "bert_attention_mask": bert_attention_mask,
            "clip_input_ids": clip_input_ids,
            "image_tensor": image_tensor,
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [ ]:
# --- 2. 专用 Dataset ---
# 注意：ResNet 的预处理与 CLIP 不同，需使用标准的 ImageNet 归一化
class SimpleFusionDataset(torch.utils.data.Dataset):
    def __init__(self, df, tokenizer, clip_processor=None):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        # 如果你没传入clip_processor，可以在这里定义或者直接用clip包
        
    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row['text_clean'])
        label = row['label']
        img_path = row['img_local_path']

        # --- 增加容错逻辑 ---
        import os
        if not os.path.exists(img_path):
            # 如果路径不存在，打印警告并返回一个全黑的图片占位符
            # print(f"Warning: File not found {img_path}") # 嫌吵可以关掉
            image_tensor = torch.zeros(3, 224, 224) 
        else:
            try:
                image = Image.open(img_path).convert('RGB')
                image_tensor = self.preprocess(image)
            except Exception as e:
                # 即使文件存在，万一文件损坏也返回空图片
                image_tensor = torch.zeros(3, 224, 224)

        # 剩下的逻辑不变
        bert_tokens = self.tokenizer(text, padding='max_length', truncation=True, max_length=128, return_tensors="pt")
        clip_tokens = clip.tokenize(text, truncate=True)

        return {
            "bert_input_ids": bert_tokens["input_ids"].flatten(),
            "bert_attention_mask": bert_tokens["attention_mask"].flatten(),
            "clip_input_ids": clip_tokens.flatten(), 
            "image_tensor": image_tensor, 
            "label": torch.tensor(label, dtype=torch.long),
            'text': text,
            'img_path': img_path,
            'orig_idx': idx  # 直接带出它在 DataFrame 里的索引
        }

In [ ]:
import pandas as pd
from torchvision import models, transforms

train_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/kaggle/input/datasets/alettatta/frd-dataset/train_ready.csv'), bert_tokenizer), batch_size=32, shuffle=True)
val_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/kaggle/input/datasets/alettatta/frd-dataset/val_ready.csv'), bert_tokenizer), batch_size=32)
test_loader = DataLoader(SimpleFusionDataset(pd.read_csv('/kaggle/input/datasets/alettatta/frd-dataset/test_ready.csv'), bert_tokenizer), batch_size=32)

In [11]:
from transformers import BertModel, BertTokenizer

# local_model_path = "/mnt/workspace/oss/yyj_ai/fake_reviews_detection/model/bert-base-chinese/"

# tokenizer = BertTokenizer.from_pretrained(local_model_path)
# model = BertModel.from_pretrained(local_model_path)

bert = BertModel.from_pretrained("bert-base-chinese")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-chinese
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# 现在可以正常 import 了
from multimodal_model import ECommerceMultimodalModel
import torch
import torch.nn as nn
from torch.optim import AdamW                 # ← 补上
from torch.optim.lr_scheduler import CosineAnnealingLR   # ← 补上

device = "cuda"
LR = 1e-4

model = ECommerceMultimodalModel(bert_model=bert).to(device)

# 优化点：引入 Label Smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# 优化点：使用 AdamW 并设置 weight_decay
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.05)

# 优化点：T_max 设置为 总迭代步数 (epochs * len(dataloader))
total_steps = EPOCHS * len(train_loader)
scheduler = CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-7)

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, scheduler, device):
    model.train()
    running_loss = 0.0

    for batch in tqdm(dataloader, desc="Training"):
        optimizer.zero_grad()

        # 统一字段名
        bert_input_ids   = batch["bert_input_ids"].to(device)
        bert_attn_mask   = batch["bert_attention_mask"].to(device)
        clip_input_ids   = batch["clip_input_ids"].to(device)
        image_tensor     = batch["image_tensor"].to(device)
        labels           = batch["label"].to(device)

        logits = model(bert_input_ids, bert_attn_mask,
                       clip_input_ids, image_tensor)
        loss = criterion(logits, labels)

        # loss.backward()
        # optimizer.step()
        # running_loss += loss.item()

        loss.backward()
        # 梯度裁剪：防止 Transformer 结构梯度爆炸
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()  # 优化点：每个 Step 更新学习率，更平滑

        running_loss += loss.item()

    return running_loss / len(dataloader)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns

@torch.no_grad()
def evaluate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_true, all_pred = [], []

    for batch in tqdm(dataloader, desc="Evaluating"):
        bert_input_ids = batch["bert_input_ids"].to(device)
        bert_attn_mask = batch["bert_attention_mask"].to(device)
        clip_input_ids = batch["clip_input_ids"].to(device)
        image_tensor   = batch["image_tensor"].to(device)
        labels = batch["label"].to(device)

        logits = model(bert_input_ids, bert_attn_mask,
                       clip_input_ids, image_tensor)
        loss = criterion(logits, labels)
        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        all_true.extend(labels.cpu().numpy())
        all_pred.extend(preds.cpu().numpy())

    acc = accuracy_score(all_true, all_pred)
    return total_loss/len(dataloader), acc, all_true, all_pred

In [ ]:
MODEL_SAVE_PATH = "/kaggle/working/multimodal_model_final.pt"
EPOCHS = 40

#### 模型训练

In [ ]:
# ---------- 早停机制配置 ----------
patience = 15
counter = 0
best_val_acc = 0 

history = {'train_loss': [], 'train_acc': [],
           'val_loss': [], 'val_acc': [],
           'val_pre_0': [], 'val_rec_0': [], 'val_f1_0': [],
           'val_pre_1': [], 'val_rec_1': [], 'val_f1_1': []}

for epoch in range(1, EPOCHS+1):
    print(f'\nEpoch {epoch}/{EPOCHS}')
    
    # ---------- 训练 ----------
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, scheduler, device)
    _, train_acc, _, _ = evaluate(model, train_loader, criterion, device) 

    # ---------- 验证 ----------
    val_loss, val_acc, val_true, val_pred = evaluate(model, val_loader, criterion, device)

    # 记录日志 (保持原有逻辑)
    report = classification_report(val_true, val_pred,
                                   target_names=['real(0)', 'fake(1)'],
                                   output_dict=True, zero_division=0)
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    for cls in ['real(0)', 'fake(1)']:
        idx = 0 if cls == 'real(0)' else 1
        history[f'val_pre_{idx}'].append(report[cls]['precision'])
        history[f'val_rec_{idx}'].append(report[cls]['recall'])
        history[f'val_f1_{idx}'].append(report[cls]['f1-score'])

    print(f'Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')

    scheduler.step()

    # ---------- 早停逻辑修改点 ----------
    if val_acc > best_val_acc:
        # 1. 如果发现更好的模型
        best_val_acc = val_acc
        torch.save(model.state_dict(), MODEL_SAVE_PATH)
        print(f'🌟 Best model saved! (Val Acc improved to {val_acc:.4f})')
        counter = 0  # 核心：一旦性能提升，计数器清零
    else:
        # 2. 如果性能没有提升
        counter += 1
        print(f'⚠️ Performance did not improve. EarlyStopping Counter: {counter}/{patience}')
        
        # 3. 达到阈值，触发早停
        if counter >= patience:
            print(f'🛑 Early stopping triggered at Epoch {epoch}. Best Val Acc reached: {best_val_acc:.4f}')
            break  # 跳出循环，结束训练

In [ ]:
def plot_training(history):
    epochs = range(1, len(history['train_loss'])+1)
    plt.figure(figsize=(15,4))

    # ---- loss / acc 曲线 ----
    plt.subplot(1,3,1)
    plt.plot(epochs, history['train_loss'], label='train loss')
    plt.plot(epochs, history['val_loss'], label='val loss')
    plt.title('Loss'); plt.legend(); plt.grid(False)

    plt.subplot(1,3,2)
    plt.plot(epochs, history['train_acc'], label='train acc')
    plt.plot(epochs, history['val_acc'], label='val acc')
    plt.title('Accuracy'); plt.legend(); plt.grid(False)

    # ---- 每类 F1 ----
    plt.subplot(1,3,3)
    plt.plot(epochs, history['val_f1_0'], label='real F1')
    plt.plot(epochs, history['val_f1_1'], label='fake F1')
    plt.title('F1 by class'); plt.legend(); plt.grid(False)

    plt.tight_layout(); plt.show()

    # ---- 混淆矩阵（最后一 epoch）----
    cm = confusion_matrix(val_true, val_pred)
    sns.heatmap(cm, annot=True, fmt='d',
                xticklabels=['real','fake'],
                yticklabels=['real','fake'],
                cmap='Blues')
    plt.ylabel('True'); plt.xlabel('Pred'); plt.title('Confusion Matrix')
    plt.show()

# 训练完成后调用
plot_training(history)

In [ ]:
test_loss, test_acc, preds, labels = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# 1. 纯手写格式，完全控制小数位
print(f"Test Loss: {test_loss:.6f} | Test Acc: {accuracy_score(labels, preds):.6f}")

# 2. 把 classification_report 转成 DataFrame 再改精度
import pandas as pd
report = classification_report(
    labels, preds,
    target_names=['real', 'fake'],
    output_dict=True          # 返回 dict 而不是字符串
)
df = pd.DataFrame(report).T   # 转置后每一行是一个类别
print(df.round(6))            # 6 位小数，也可以改成 8、10 …

#### 不同genre数据评测

In [6]:
import torch
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, accuracy_score

# 1. 加载所有原始数据并合并
# 既然你需要按 Genre 查看全貌，就把三个表拼在一起
print("正在加载并合并数据...")
df_train = pd.read_pickle('/kaggle/working/tmp/train_processed.pkl')
df_val = pd.read_pickle('/kaggle/working/tmp/val_processed.pkl')
df_test = pd.read_pickle('/kaggle/working/tmp/test_processed.pkl')

# 合并为一个大表，专门用于各领域性能测试
df_all = pd.concat([df_train, df_val, df_test], axis=0).reset_index(drop=True)
print(f"数据合并完成，总样本数: {len(df_all)}")

正在加载并合并数据...
数据合并完成，总样本数: 7790


In [7]:
df_all

,user_id,review_time,image_url,review_text,label,genre,text_clean,img_local_path
0,m**3,2025年11月20日,//img.alicdn.com/imgextra/i2/46116860184273849...,整体评价： 保湿控油效果：好，质量好，服務各方面都好，值得购买,0,美妆,整体评价 保湿控油效果 好，质量好，服务各方面都好，值得购买,/kaggle/working/images/c6352cc68309c3f922d3f21...
1,i**?,2025年10月31日,//img.alicdn.com/imgextra/i3/46116860184273802...,常年买这款钙片，孕期吃，家里老人也在吃，迷你钙比较好吞服！补充钙的同时也补维生素D，促进吸收...,1,保健,常年买这款钙片，孕期吃，家里老人也在吃，迷你钙比较好吞服！补充钙的同时也补维生素D，促进吸收...,/kaggle/working/images/8f9daea14fb3c7f3e942d66...
2,一**火,2025年11月12日,//img.alicdn.com/imgextra/i4/46116860184273878...,喝酒熬夜必备，产品不错，性价比高，第二次回购了,0,保健,喝酒熬夜必备，产品不错，性价比高，第二次回购了,/kaggle/working/images/10ebf6f3cd91328afd106e1...
3,公**9,2025年10月30日,//img.alicdn.com/imgextra/i3/46116860184273829...,霜很水润，给婆婆买的，一直用这款还是一如既往的好用,0,美妆,霜很水润，给婆婆买的，一直用这款还是一如既往的好用,/kaggle/working/images/5431855895cbdd81d74777c...
4,瑟***,2025年9月14日,//img.alicdn.com/imgextra/i3/46116860184273864...,用玉兰油已经多年，然，仍然觉得她是性价比最高的基础保湿面霜，敏感肌在换季时维稳也很躜,1,美妆,用玉兰油已经多年，然，仍然觉得她是性价比最高的基础保湿面霜，敏感肌在换季时维稳也很躜,/kaggle/working/images/8f3923e12f7ff9e81ae9054...
...,...,...,...,...,...,...,...,...
7785,Z***女,05-20,https://img30.360buyimg.com/shaidan/s300x300_j...,这款防晒霜安耐晒还是我2021年练车的时候第一次买，然后这几年的防晒霜一直都是用这个品牌的，...,0,美妆,这款防晒霜安耐晒还是我2021年练车的时候第一次买，然后这几年的防晒霜一直都是用这个品牌的，...,/kaggle/working/images/95d231aba18cbe1bcb34b10...
7786,小**8,2025年9月7日,//img.alicdn.com/imgextra/i2/O1CN01Dqo1gd1wMhn...,#赫莲娜新品黑绷带真心话# 情人节特意为爱海钓的老公准备了赫莲娜黑绷带面霜，他每次出海回来皮...,1,美妆,情人节特意为爱海钓的老公准备了赫莲娜黑绷带面霜，他每次出海回来皮肤都又红又干，晒伤明显。自从...,/kaggle/working/images/cfc96ca98c1e34d173f709c...
7787,匿名买家,2023年12月25日,//img.alicdn.com/imgextra/i2/0/O1CN01aZYLnI1IZ...,物流很快， 但是真的不好用，味道太浓，而且很难推开,0,美妆,物流很快， 但是真的不好用，味道太浓，而且很难推开,/kaggle/working/images/2d7e18bec4d7d48bffa6cdc...
7788,木***-,2018-07-17,https://img30.360buyimg.com/shaidan/s300x300_j...,颜色很好看，上色也不错。很小巧，方便携带。买了两个送给朋友一个。赞。,0,美妆,颜色很好看，上色也不错。很小巧，方便携带。买了两个送给朋友一个。赞。,/kaggle/working/images/6edb33e4e2ff17a69e2ca70...


In [8]:
import pandas as pd
import torch

# 1. 确保所有数据已合并
df_all = pd.concat([df_train, df_val, df_test], axis=0).reset_index(drop=True)

# 2. 检查并强制转换标签
# 打印一下，看看有没有奇怪的值（比如 -1, 2, 或者 '0' 这种字符串）
print("原始标签分布:", df_all['label'].unique())

# 强制转换为 int，并只保留 0 和 1
df_all['label'] = pd.to_numeric(df_all['label'], errors='coerce') # 转为数字，非法字符转为 NaN
df_all = df_all.dropna(subset=['label']) # 删掉无法转换的行
df_all['label'] = df_all['label'].astype(int) # 转为整数

# 核心过滤：只留 0 和 1
df_all = df_all[df_all['label'].isin([0, 1])].reset_index(drop=True)

print("清洗后标签分布:", df_all['label'].unique())
print(f"最终样本数: {len(df_all)}")

原始标签分布: [0 1]
清洗后标签分布: [0 1]
最终样本数: 7790


In [20]:
class MultimodalReviewDataset(Dataset):
    def __init__(self, dataframe, bert_tokenizer, clip_processor, image_transform):
        self.df = dataframe.reset_index(drop=True)
        self.bert_tokenizer = bert_tokenizer
        self.clip_processor = clip_processor
        self.image_transform = image_transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["text_clean"])

        # ✅ 1. BERT tokenizer（给 BERT 用）
        bert_inputs = self.bert_tokenizer(
            text,
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
        )

        # ✅ 2. CLIP tokenizer（给 CLIP 用）
        clip_inputs = self.clip_processor(
            text=text,
            truncation=True,
            max_length=77,
            padding="max_length",
            return_tensors="pt"
        )

        # ✅ 图像
        img_tensor = load_and_transform_image(
            row["img_local_path"], 
            self.image_transform
        )

        label = torch.tensor(row["label"], dtype=torch.long)

        return {
            "bert_input_ids": bert_inputs["input_ids"].squeeze(0),
            "bert_attention_mask": bert_inputs["attention_mask"].squeeze(0),
            "clip_input_ids": clip_inputs["input_ids"].squeeze(0),
            "image": img_tensor,
            "label": label
        }

In [23]:
from transformers import BertTokenizer, CLIPProcessor

# ✅ 必须和训练时一致（很重要）
bert_tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

clip_processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

In [28]:
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, classification_report
import os

# 👉 关键：让 CUDA 报错定位准确
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

MODEL_PATH = "/kaggle/working/multimodal_model_final.pt"

# ✅ 1. 初始化模型（必须和训练时完全一致）
model = ECommerceMultimodalModel(bert_model=bert).to(device)

# ✅ 2. 加载权重（推荐安全写法）
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)

model.eval()

print("✅ 模型加载成功")

# ----------------------------------------------------
# ✅ 3. 单个 genre 测试函数（稳定版）
# ----------------------------------------------------
def evaluate_genre(model, df, genre):

    print(f"\n{'='*40}")
    print(f"📊 评估领域: {genre}")
    print(f"{'='*40}")

    # ✅ 筛选数据
    sub_df = df[df['genre'] == genre].copy()

    if len(sub_df) == 0:
        print("❌ 没有数据")
        return

    # ✅ 保证 label 合法（非常重要）
    sub_df['label'] = sub_df['label'].astype(int)
    sub_df = sub_df[sub_df['label'].isin([0,1])]

    print("样本数:", len(sub_df))
    print("label分布:\n", sub_df['label'].value_counts())

    # ✅ Dataset & Loader
    dataset = MultimodalReviewDataset(
        sub_df,
        bert_tokenizer,
        clip_processor,
        image_transform
    )
    loader = DataLoader(dataset, batch_size=32, shuffle=False)

    all_preds = []
    all_labels = []

    # ------------------------------------------------
    # ✅ 只做推理（不训练）
    # ------------------------------------------------
    with torch.no_grad():
        for batch in loader:

            bert_ids = batch["bert_input_ids"].to(device)
            bert_mask = batch["bert_attention_mask"].to(device)
            clip_ids = batch["clip_input_ids"].to(device)
            bert_mask = batch["bert_attention_mask"].to(device)
            images = batch["image"].to(device)
            labels = batch["label"].to(device)

            # 👉 关键：安全 CLIP 输入（避免炸）
            bs = bert_ids.size(0)
            clip_ids = torch.zeros((bs, 77), dtype=torch.long).to(device)

            logits = model(
                bert_input_ids=bert_ids,
                bert_attention_mask=bert_mask,
                clip_input_ids=clip_ids,
                image_tensor=images
            )

            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # ------------------------------------------------
    # ✅ 输出结果
    # ------------------------------------------------
    acc = accuracy_score(all_labels, all_preds)

    print(f"\n✅ Accuracy: {acc:.4f}")
    print("\n📋 分类报告:")
    print(classification_report(all_labels, all_preds, digits=4))


# ----------------------------------------------------
# ✅ 4. 循环所有 genre
# ----------------------------------------------------
genres = df_all['genre'].unique()

for g in genres:
    evaluate_genre(model, df_all, g)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
/tmp/ipykernel_941/779578250.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more detai

✅ 模型加载成功

📊 评估领域: 美妆
样本数: 2673
label分布:
 label
0    1736
1     937
Name: count, dtype: int64


/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



✅ Accuracy: 0.7987

📋 分类报告:
              precision    recall  f1-score   support

           0     0.8407    0.8514    0.8460      1736
           1     0.7180    0.7012    0.7095       937

    accuracy                         0.7987      2673
   macro avg     0.7794    0.7763    0.7778      2673
weighted avg     0.7977    0.7987    0.7982      2673


📊 评估领域: 保健
样本数: 4501
label分布:
 label
1    2404
0    2097
Name: count, dtype: int64

✅ Accuracy: 0.9120

📋 分类报告:
              precision    recall  f1-score   support

           0     0.9083    0.9022    0.9053      2097
           1     0.9152    0.9205    0.9179      2404

    accuracy                         0.9120      4501
   macro avg     0.9118    0.9114    0.9116      4501
weighted avg     0.9120    0.9120    0.9120      4501


📊 评估领域: 母婴
样本数: 616
label分布:
 label
0    357
1    259
Name: count, dtype: int64

✅ Accuracy: 0.6818

📋 分类报告:
              precision    recall  f1-score   support

           0     0.6746    0.8711    0.